In [2]:
import os
import numpy as np
import pandas as pd

os.chdir(r"C:\Users\ricky\Documents\ai-training-week1\Week 1- Day 5")

df = pd.read_csv("clean_data.csv")
df["join_date"] = pd.to_datetime(df["join_date"])

reference_date = df["join_date"].max()
df["tenure_days"] = (reference_date - df["join_date"]).dt.days
df["join_date_missing_int"] = df["join_date_missing"].astype(int)
df["is_salary_outlier_int"] = df["is_salary_outlier"].astype(int)

numeric_cols = ["salary", "tenure_days"]
summary_stats = df[numeric_cols].agg(["mean", "median", "min", "max", "std"]).round(2)
print("=== Summary Statistics ===")
print(summary_stats.to_string())

dept_summary = (
    df.groupby("department", observed=False)
    .agg(
        employee_count=("employee_id", "count"),
        avg_salary=("salary", "mean"),
        median_salary=("salary", "median"),
        min_salary=("salary", "min"),
        max_salary=("salary", "max"),
        avg_tenure_days=("tenure_days", "mean"),
    )
    .round(2)
)
print("\n=== Group Summary by Department ===")
print(dept_summary.to_string())

corr_cols = ["salary", "tenure_days", "join_date_missing_int", "is_salary_outlier_int"]
corr_matrix = df[corr_cols].corr(numeric_only=True).round(3)
print("\n=== Correlation Matrix ===")
print(corr_matrix.to_string())

corr_pairs = []
cols = corr_matrix.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        val = corr_matrix.iloc[i, j]
        if pd.notna(val):
            corr_pairs.append((cols[i], cols[j], val, abs(val)))

corr_pairs.sort(key=lambda x: x[3], reverse=True)
print("\n=== Top 3 Numeric Relationships (by |r|) ===")
for rank, (c1, c2, r_val, _) in enumerate(corr_pairs[:3], start=1):
    print(f"{rank}. {c1} <-> {c2}: r = {r_val:.3f}")

dept_budget = pd.DataFrame({
    "department": ["Sales", "Engineering", "HR"],
    "annual_budget_inr": [400000, 900000, 300000],
})

merged_df = pd.merge(df, dept_budget, on="department", how="left")
budget_check = (
    merged_df.groupby("department", observed=False)
    .agg(
        headcount=("employee_id", "count"),
        total_salary=("salary", "sum"),
        annual_budget_inr=("annual_budget_inr", "first"),
    )
)
budget_check["budget_utilization_pct"] = (
    (budget_check["total_salary"] / budget_check["annual_budget_inr"]) * 100
).round(2)

print("\n=== Merged Department Budget vs. Actual Salary Spend ===")
print(budget_check.to_string())

q1_avg_salary = df["salary"].mean()
q1_med_salary = df["salary"].median()

q2_top_dept = dept_summary["avg_salary"].idxmax()
q2_top_dept_sal = dept_summary.loc[q2_top_dept, "avg_salary"]
q2_low_dept = dept_summary["avg_salary"].idxmin()
q2_low_dept_sal = dept_summary.loc[q2_low_dept, "avg_salary"]

q3_outlier_count = int(df["is_salary_outlier"].sum())
q3_outlier_names = df.loc[df["is_salary_outlier"], "name"].tolist()

q4_missing_date_count = int(df["join_date_missing"].sum())

q5_eng_util = budget_check.loc["Engineering", "budget_utilization_pct"]
q5_sales_util = budget_check.loc["Sales", "budget_utilization_pct"]

print("\n=== Business Questions & Answers ===")
print(f"Q1. What are the average and median salaries across cleaned employees? -> Mean: {q1_avg_salary:,.2f} INR | Median: {q1_med_salary:,.2f} INR")
print(f"Q2. Which department has the highest vs. lowest average salary? -> Highest: {q2_top_dept} ({q2_top_dept_sal:,.2f} INR) vs. Lowest: {q2_low_dept} ({q2_low_dept_sal:,.2f} INR)")
print(f"Q3. How many salary outliers exist via IQR, and who are they? -> {q3_outlier_count} outlier ({', '.join(q3_outlier_names)})")
print(f"Q4. How many employees had missing/invalid join dates flagged? -> {q4_missing_date_count} employee(s)")
print(f"Q5. How much of department budget is used by salaries in Engineering vs. Sales? -> Engineering: {q5_eng_util}% | Sales: {q5_sales_util}%")

pivot_tbl = pd.pivot_table(
    merged_df,
    values="salary",
    index="department",
    columns="is_salary_outlier",
    aggfunc="count",
    fill_value=0,
    observed=False,
)
print("\n=== Bonus: Pivot Table (Department vs. Salary Outlier Count) ===")
print(pivot_tbl.to_string())


=== Summary Statistics ===
           salary  tenure_days
mean    102700.00       104.40
median   68500.00       106.00
min      58000.00         0.00
max     250000.00       203.00
std      82505.76        77.58

=== Group Summary by Department ===
             employee_count  avg_salary  median_salary  min_salary  max_salary  avg_tenure_days
department                                                                                     
Engineering               3   130166.67        72000.0     68500.0    250000.0             71.0
Sales                     2    61500.00        61500.0     58000.0     65000.0            154.5

=== Correlation Matrix ===
                       salary  tenure_days  join_date_missing_int  is_salary_outlier_int
salary                  1.000       -0.748                 -0.303                  0.998
tenure_days            -0.748        1.000                  0.012                 -0.752
join_date_missing_int  -0.303        0.012                  1.000      

## Executive Summary: Employee Compensation & Budget Analysis

### 1. Problem
HR and finance teams need an accurate view of department-level compensation, tenure distribution, data anomalies, and annual salary budget utilization across employees—starting from raw HR records containing formatting issues, missing values, duplicates, and extreme salary entries.

### 2. Data
* **Source & Cleaning:** Raw employee records (`raw_data.csv`) were standardized in `data_cleaning.py` into `clean_data.csv` (**5 valid employees** across **Engineering** and **Sales**, **0 nulls**, **0 duplicates**).
* **Transformations & Enrichment:** Trimmed whitespace, standardized casing, converted numeric/date types, imputed missing numeric/categorical fields with median/mode, flagged 1 invalid join date (`Karthik Rao`) and 1 IQR salary outlier (`Sanjay Verma`, `250,000.00 INR`), and merged department budget figures (`900,000 INR` Engineering; `400,000 INR` Sales).

### 3. Key Insights
1. **Compensation Skew Driven by One Outlier:** Overall mean salary is **102,700.00 INR**, significantly higher than the median of **68,500.00 INR** (`std = 82,505.76 INR`), because a single Engineering outlier (**Sanjay Verma** at **250,000.00 INR**) pulls the average upward.
2. **Engineering Leads Spend & Average Pay:** Engineering averages **130,166.67 INR** across 3 employees (median **72,000.00 INR**), more than double Sales' average of **61,500.00 INR** across 2 employees (median **61,500.00 INR**).
3. **Healthy Budget Headroom Across Both Departments:** Engineering utilizes **43.39%** of its annual salary budget (**390,500.00 INR** of **900,000 INR**), while Sales utilizes **30.75%** (**123,000.00 INR** of **400,000 INR**), leaving over half of allocated compensation budgets available.

### 4. Recommendations
1. **Use Median Over Mean for Compensation Benchmarking & Audit the Outlier:** Base future salary band reviews on department medians (**72,000 INR** Engineering; **61,500 INR** Sales) rather than skewed averages, and verify with HR whether Sanjay Verma's **250,000 INR** package reflects a senior/executive role or a payroll entry error.
2. **Reallocate Unused Budget Headroom Toward Hiring & Validation:** With **56.61%** of Engineering's budget and **69.25%** of Sales' budget unspent, prioritize planned headcount expansion while enforcing automated date/salary validation rules at data entry to prevent `invalid_date` and formatting errors.

### 5. Limitations
* **Small Sample Size (`n = 5`):** With only 5 post-cleaning records (3 in Engineering, 2 in Sales), correlation metrics (e.g., `salary` vs. `tenure_days` at `r = -0.748`) and group averages are highly sensitive to single individuals.
* **Static Snapshot & Imputed Dates:** Tenure calculations rely on a fixed reference date and include one median-imputed join date (`Karthik Rao`), which slightly affects tenure-based correlations.